# Settings

In [1]:
library(dplyr)
library(rtracklayer)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, saveRDS, setdiff,
    table, tapply, union, unique, unsplit, which.max, which.min


Load

In [2]:
# Input files:

# - GENCODE reference file
gencode_file <- "../input_data/GENCODE/gencode.v47.annotation.gtf.gz"

# - FUSIL file
fusil_file <- "../input_data/FUSIL/fusil_all_260724.txt"

# Output file
outfile <- "rowdata_human.rds"

In [3]:
sessionInfo()

R version 4.4.2 (2024-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 22.04.3 LTS

Matrix products: default
BLAS/LAPACK: /mnt/array/process/dgorkin/.conda/envs/manuscript_fusil/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
[1] rtracklayer_1.66.0   GenomicRanges_1.58.0 GenomeInfoDb_1.42.0 
[4] IRanges_2.40.0       S4Vectors_0.44.0     BiocGenerics_0.52.0 
[7] dplyr_1.1.4         

loaded via a namespace (and not attached):
 [1] generics_0.1.3              SparseArray_1.6.0          

# Load and format GENCODE reference

In [4]:
gencode <- import(gencode_file)

# subset gtf entries for genes
gencode <- gencode[which(gencode$type=="gene"),]

# create rowdata data frame
rowdata <- as.data.frame(gencode@elementMetadata)

# set rownames to gene ids
rownames(rowdata) <- rowdata$gene_id

# create column with "base" gene id (without the decimal gene version number)
rowdata$gene_id_base <- rownames(rowdata) %>%
    lapply(FUN=strsplit, split="\\.") %>%
    unlist() %>%
    matrix(ncol=2, byrow=T)  %>%
    subset.matrix(select = 1) %>%
    as.character()

#  eliminate columns with all NAs
rowdata <- rowdata[,colSums(is.na(rowdata))<nrow(rowdata)]

# add genome coordinates
rowdata$chr <- as.character(gencode@seqnames)
rowdata$start <- as.numeric(gencode@ranges@start)
rowdata$end <- rowdata$start + as.numeric(gencode@ranges@width)
#rm(gencode)

# view summary
print(paste("## Genes loaded: ", nrow(rowdata)))

[1] "## Genes loaded:  78724"


In [5]:
# Filter out pseudogenes
temp.filt <- grepl("pseudogene", rowdata$gene_type, ignore.case=TRUE)
print(paste("## Filter pseudogenes: ", length(which(temp.filt))))
rowdata <- rowdata[which(!temp.filt),]
print(paste("## Genes remaining: ", nrow(rowdata)))

[1] "## Filter pseudogenes:  15200"
[1] "## Genes remaining:  63524"


In [6]:
# Filter entries without valid hgnc_id (to match fusil), gene_id (to match rna-seq), and gene_names
temp.filt <- is.na(rowdata$hgnc_id) | is.na(rowdata$gene_id) | is.na(rowdata$gene_name)
print(paste("## Filter genes without valid identifiers: ", length(which(temp.filt))))
rowdata <- rowdata[which(!temp.filt),]
print(paste("## Genes remaining: ", nrow(rowdata)))

[1] "## Filter genes without valid identifiers:  32740"
[1] "## Genes remaining:  30784"


In [7]:
# confirm all gene_ids are unique.
!any(duplicated(rowdata$gene_id))

[1] TRUE

In [8]:
# confirm all gene_ids are unique. If not, will remove duplicate entries
!any(duplicated(rowdata$hgnc_id))

temp.filt <- rowdata$hgnc_id %in% rowdata$hgnc_id[which(duplicated(rowdata$hgnc_id))]
print(paste("## Filter entries with duplicate hgnc_id: ", length(which(temp.filt))))
rowdata <- rowdata[which(!temp.filt),]
print(paste("## Genes remaining: ", nrow(rowdata)))
!any(duplicated(rowdata$hgnc_id))

[1] FALSE

[1] "## Filter entries with duplicate hgnc_id:  88"
[1] "## Genes remaining:  30696"


[1] TRUE

In [9]:
# confirm all gene_names are unique.
!any(duplicated(rowdata$gene_name))

[1] TRUE

# Load FUSIL annotations

In [10]:
fusil <- read.delim(fusil_file, as.is=T, sep = "\t", header = T)
fusil[fusil == '-'] <- NA

# view summary
print(paste("## FUSIL calls loaded: ", nrow(fusil)))

[1] "## FUSIL calls loaded:  8037"


In [11]:
# Filter entries without valid hgnc_id (to match fusil)
temp.filt <- is.na(fusil$hgnc_id) 
print(paste("## Filter entries without valid identifiers: ", length(which(temp.filt))))
fusil <- fusil[which(!temp.filt),]
print(paste("## Entries remaining: ", nrow(fusil)))

[1] "## Filter entries without valid identifiers:  595"
[1] "## Entries remaining:  7442"


In [12]:
# confirm all gene_ids are unique.
!any(duplicated(rowdata$hgnc_id))

[1] TRUE

In [13]:
# confirm all gene_ids are unique.
!any(duplicated(rowdata$gene_symbol))

[1] TRUE

# Combine tables

In [14]:
# check if all fusil hgnc_id in rowdata reference. If not, how many are missing
all(fusil$hgnc_id %in% rowdata$hgnc_id)

temp.missing <- which(!fusil$hgnc_id %in% rowdata$hgnc_id)
print(paste("## FUSIL hgnc_id missing in reference: ", length(temp.missing)))
print(fusil$gene_symbol[temp.missing])

[1] FALSE

[1] "## FUSIL hgnc_id missing in reference:  8"
[1] "GTPBP6"    "CRLF2"     "LINC02491" "FBXL21P"   "PRSS45P"   "GLRA4"    
[7] "IL3RA"     "IL9R"     


In [15]:
rowdata <- merge(x = rowdata,
                 y = fusil,
                 by = "hgnc_id",
                 all.x = TRUE,
                 sort = FALSE) %>%
    arrange(gene_name) %>%
    `rownames<-`(.[,"hgnc_id"]) %>%
    select(-hgnc_id)

In [16]:
print(paste("## Final genes: ", nrow(rowdata)))

[1] "## Final genes:  30696"


In [17]:
saveRDS(rowdata, file=outfile)